# Scanner Evaluation

Configurable analysis workflow for single-scan evaluation or multi-scan comparisons across scanners, intended for running on a single benchmark.

The goal of this script is to provide various descriptive plots and stats for evaluating the performance of a scanner, and includes comparison to human graded or pre-defined (synthetic) measures of desired performance. 

Configure the notebook by setting the input paths for scan results and the three preprocessed CSVs, then adjust the analysis options in the top config block. The main knobs are the violation threshold, optional subset or scanner_source include/exclude filters, display order for scanners and human label sets, explicit scanner-to-human label mappings, and TARGET_RULES, which define the ground truth for each subset either from a human label column or a uniform 0%/100% violation assumption. Any scanner, human label, or subset referenced in those mappings must exist in the loaded data or the notebook will report it as missing.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from scan_utils import (
    add_violation_flags,
    build_analysis_frame,
    build_comparison_table,
    build_human_analysis_frame,
    format_metric_percent,
    load_eval_runs_csv,
    load_human_labels_long,
    load_human_labels_wide,
    load_scan_results,
    summarize_coverage,
    summarize_grade_distribution,
    summarize_performance_metrics,
    summarize_scanner_comparison,
    summarize_violation_rates,
)


In [ ]:
# Configuration
SCAN_RESULTS_DIR = Path("../analysis/data") #will pull all scans from this directory
EVAL_RUNS_CSV = Path("preprocessed_eval_runs.csv")
HUMAN_LABELS_WIDE_CSV = Path("preprocessed_human_labels.csv")
HUMAN_LABELS_LONG_CSV = Path("preprocessed_human_labels_long.csv")

#threshold for labeling a transcript as 'violating'. This could be tuned to be more or less sensitive. Used for things like calculating sensitivity/specificicity.
VIOLATION_THRESHOLD = 2

# Specifying scanners to include
# Can set to none to include everything within the scan_results_dir (recursively includes subfolders)
INCLUDE_SCANNER_SOURCES = None #specific scanner IDs, eg "scan_id=XyYqreMfYtSrPcbhEG2wxB" 
EXCLUDE_SCANNER_SOURCES = None 
INCLUDE_SUBSETS = None #directory level subsets, eg "synth/t5-web"
EXCLUDE_SUBSETS = None

# Display labels for scanner sources (the scan_id directories).
# Set to None to auto-number sources as "scan1", "scan2", etc. in load order.
# Set to a dict mapping scanner_source values to friendly names, eg:
#   {"scan_id=XyYqreMfYtSrPcbhEG2wxB": "baseline", "scan_id=abc123": "v2"}
SCANNER_SOURCE_LABELS = None

# taken from scanner descriptions
SCANNER_DISPLAY_ORDER = ["answer_format", "ground_truth_access"]
HUMAN_DISPLAY_ORDER = ["oh1_JM", "t5_JM"]

#matching between scanner labels and human grades, for comparing
SCANNER_TO_HUMAN_LABEL = {
    "answer_format": "oh1_JM",
    "ground_truth_access": "t5_JM",
}

# Specify the targets for comparison of scanner accuracy and other performance metrics against some target grades (note this is always converting to binary labeling based on the threshold value set above)
# Can use either human labeled data or set a uniform expectation, intended for synth data that is assumed to have 100% violation rates.
# Subsets omitted from TARGET_RULES are excluded from performance metrics.
TARGET_RULES = {
    "default": {
        "mode": "human",
        "label_name": "oh1_JM",
        "scanner_overrides": {
            "ground_truth_access": {"mode": "human", "label_name": "t5_JM"},
        },
    },
    "synth/t5-contamination": {
        "mode": "uniform",
        "positive_rate": 1.0,
    },
    "synth/t5-web": {
        "mode": "uniform",
        "positive_rate": 1.0,
    },
}

In [ ]:
eval_runs = load_eval_runs_csv(EVAL_RUNS_CSV)
human_labels_wide = load_human_labels_wide(HUMAN_LABELS_WIDE_CSV)
human_labels_long = load_human_labels_long(HUMAN_LABELS_LONG_CSV)
scans = load_scan_results(SCAN_RESULTS_DIR)

analysis = build_analysis_frame(
    scans,
    eval_runs,
    human_labels_long=human_labels_long,
    include_scanner_sources=INCLUDE_SCANNER_SOURCES,
    exclude_scanner_sources=EXCLUDE_SCANNER_SOURCES,
    include_subsets=INCLUDE_SUBSETS,
    exclude_subsets=EXCLUDE_SUBSETS,
)
human_analysis = build_human_analysis_frame(
    eval_runs,
    human_labels_long,
    include_subsets=INCLUDE_SUBSETS,
    exclude_subsets=EXCLUDE_SUBSETS,
)

# Relabel scanner sources for cleaner plot labels
if SCANNER_SOURCE_LABELS is not None:
    _source_label_map = SCANNER_SOURCE_LABELS
else:
    _unique_sources = analysis["scanner_source"].dropna().unique().tolist()
    _source_label_map = {src: f"scan{i+1}" for i, src in enumerate(sorted(_unique_sources))}
analysis["scanner_source"] = analysis["scanner_source"].map(_source_label_map).fillna(analysis["scanner_source"])

available_scanners = analysis["scanner_key"].dropna().unique().tolist()
scanner_keys = [key for key in SCANNER_DISPLAY_ORDER if key in available_scanners]
scanner_keys += [key for key in sorted(available_scanners) if key not in scanner_keys]

available_human_labels = set(human_labels_wide.columns)
human_labels_for_display = [
    label for label in HUMAN_DISPLAY_ORDER if label in available_human_labels
]
human_labels_for_display += [
    label
    for label in sorted(set(SCANNER_TO_HUMAN_LABEL.values()))
    if label in available_human_labels and label not in human_labels_for_display
]

comparison = build_comparison_table(
    analysis,
    human_labels_wide=human_labels_wide,
    human_label_names=human_labels_for_display,
)

coverage = summarize_coverage(
    analysis,
    human_labels_wide=human_labels_wide,
    human_label_names=human_labels_for_display,
)

missing_scanners = [key for key in SCANNER_DISPLAY_ORDER if key not in available_scanners]
missing_human_labels = [label for label in HUMAN_DISPLAY_ORDER if label not in available_human_labels]
missing_target_subsets = [subset for subset in TARGET_RULES if subset not in set(analysis["eval_subset"].dropna())]
missing_target_labels = []
for subset, rule in TARGET_RULES.items():
    rules_to_check = [rule] + list(rule.get("scanner_overrides", {}).values())
    for item in rules_to_check:
        label_name = item.get("label_name")
        if item.get("mode") == "human" and label_name not in available_human_labels:
            missing_target_labels.append((subset, label_name))

print(f"Scan rows: {len(analysis):,}")
print(f"Unique transcripts: {analysis['transcript_id'].nunique():,}")
print(f"Scanner sources: {analysis['scanner_source'].nunique()}")
print(f"Scanners: {scanner_keys}")
print(f"Scanner source labels: {dict(sorted(_source_label_map.items(), key=lambda x: x[1]))}")
print(f"Human labels in use: {human_labels_for_display}")
if missing_scanners:
    print(f"Missing configured scanners: {missing_scanners}")
if missing_human_labels:
    print(f"Missing configured human labels: {missing_human_labels}")
if missing_target_subsets:
    print(f"Target subsets not present in filtered data: {missing_target_subsets}")
if missing_target_labels:
    print(f"Target rules reference missing human labels: {missing_target_labels}")

display(coverage)
display(comparison.head())

In [ ]:
score_colors = {0: "#4393c3", 1: "#f4d35e", 2: "#d1495b", 3: "#7f0000"}

def _subset_order(values):
    values = [value for value in values if pd.notna(value)]
    default_first = [value for value in values if value == "default"]
    rest = sorted(value for value in values if value != "default")
    return default_first + rest


def _plot_stacked_bars(ax, labels, rows, grade_levels):
    """Plot stacked grade bars on an axes. Grade 3 at bottom, 0 at top."""
    if not rows:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        return
    plot_df = pd.DataFrame(rows, index=labels).fillna(0)
    bottom = np.zeros(len(plot_df))
    x = np.arange(len(plot_df))
    for grade in reversed(grade_levels):
        if grade in plot_df.columns:
            values = plot_df[grade].to_numpy()
        else:
            values = np.zeros(len(plot_df))
        ax.bar(x, values, bottom=bottom, label=str(grade), color=score_colors.get(grade, "#999999"))
        bottom += values
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index, rotation=45, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Proportion")


def plot_grade_distributions(
    analysis,
    human_analysis,
    scanner_keys,
    human_labels_for_display,
    scanner_to_human_label=None,
    include_source=False,
):
    """Plot grade distributions by benchmark with scanners as subplots.

    Matched human labels (via scanner_to_human_label) appear as the first bar
    within each scanner's subplot for each eval_subset.
    """
    scanner_to_human_label = scanner_to_human_label or {}
    benchmarks = sorted(analysis["benchmark"].dropna().unique().tolist())

    for benchmark in benchmarks:
        bench_analysis = analysis[analysis["benchmark"] == benchmark]
        bench_human = human_analysis[human_analysis["benchmark"] == benchmark]

        scanner_panels = [
            key for key in scanner_keys
            if not bench_analysis[bench_analysis["scanner_key"] == key].empty
        ]
        if not scanner_panels:
            continue

        n_panels = len(scanner_panels)
        fig, axes = plt.subplots(n_panels, 1, figsize=(8, 5 * n_panels), squeeze=False)
        axes = axes[:,0]
        subsets = _subset_order(bench_analysis["eval_subset"].dropna().unique().tolist())

        for panel_idx, key in enumerate(scanner_panels):
            scanner_data = bench_analysis[bench_analysis["scanner_key"] == key]
            scanner_dist = summarize_grade_distribution(
                scanner_data,
                group_cols=["eval_subset"] + (["scanner_source"] if include_source else []),
            )

            # Look up matched human label
            human_label = scanner_to_human_label.get(key)
            human_dist = None
            if human_label:
                human_data = bench_human[bench_human["label_name"] == human_label]
                if not human_data.empty:
                    human_dist = summarize_grade_distribution(
                        human_data, group_cols=["eval_subset"], grade_col="target_num",
                    )

            # Collect all grade levels across human + scanner
            grade_levels = sorted(set(scanner_dist["grade"].dropna().astype(int).unique()) | (
                set(human_dist["grade"].dropna().astype(int).unique()) if human_dist is not None else set()
            ))

            # Determine scanner source values
            source_values = sorted(scanner_data["scanner_source"].dropna().unique().tolist())
            has_multiple_sources = include_source and len(source_values) > 1

            labels = []
            rows = []
            for subset in subsets:
                subset_scanner = scanner_dist[scanner_dist["eval_subset"] == subset]
                human_piece = None
                if human_dist is not None:
                    human_piece = human_dist[human_dist["eval_subset"] == subset]

                # Skip subset if both human and scanner have no data
                if subset_scanner.empty and (human_piece is None or human_piece.empty):
                    continue

                # Human bar first (if matched and has data for this subset)
                if human_piece is not None and not human_piece.empty:
                    piece = human_piece
                    row = {g: 0 for g in grade_levels}
                    for _, item in piece.iterrows():
                        row[int(item["grade"])] = item["proportion"]
                    rows.append(row)
                    labels.append(f"{subset}\n{human_label}")

                # Scanner bar(s)
                if has_multiple_sources:
                    for source in source_values:
                        piece = subset_scanner[subset_scanner["scanner_source"] == source]
                        row = {g: 0 for g in grade_levels}
                        for _, item in piece.iterrows():
                            row[int(item["grade"])] = item["proportion"]
                        rows.append(row)
                        labels.append(f"{subset}\n{source}")
                elif human_dist is not None:
                    # Single source, but label needed to distinguish from human
                    source_label = source_values[0] if source_values else key
                    row = {g: 0 for g in grade_levels}
                    for _, item in subset_scanner.iterrows():
                        row[int(item["grade"])] = item["proportion"]
                    rows.append(row)
                    labels.append(f"{subset}\n{source_label}")
                else:
                    row = {g: 0 for g in grade_levels}
                    for _, item in subset_scanner.iterrows():
                        row[int(item["grade"])] = item["proportion"]
                    rows.append(row)
                    labels.append(subset)

            _plot_stacked_bars(axes[panel_idx], labels, rows, grade_levels)
            axes[panel_idx].set_title(key)

        handles, legend_labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, legend_labels, title="Grade", loc="upper right", bbox_to_anchor=(1.02, 0.95))
        fig.suptitle(f"Grade Distribution — {benchmark}", fontsize=14)
        fig.tight_layout(rect=[0, 0, 0.95, 0.93])
        plt.show()


def plot_rate_summary(rate_df, title, series_col, rate_col="violation_rate", subgroup_col=None):
    if rate_df.empty:
        print(f"No data for {title}")
        return

    if "benchmark" in rate_df.columns:
        benchmarks = sorted(rate_df["benchmark"].dropna().unique().tolist())
    else:
        benchmarks = [None]

    for benchmark in benchmarks:
        if benchmark is not None:
            bench_df = rate_df[rate_df["benchmark"] == benchmark]
            full_title = f"{title} — {benchmark}"
        else:
            bench_df = rate_df
            full_title = title

        subsets = _subset_order(bench_df["eval_subset"].dropna().unique().tolist())
        series_values = sorted(bench_df[series_col].dropna().unique().tolist())
        source_values = [None]
        if subgroup_col and subgroup_col in bench_df.columns and bench_df[subgroup_col].nunique() > 1:
            source_values = sorted(bench_df[subgroup_col].dropna().unique().tolist())

        width = 0.8 / max(1, len(series_values) * len(source_values))
        x = np.arange(len(subsets))
        fig, ax = plt.subplots(figsize=(max(8, len(subsets) * 1.8), 5))
        offset = 0
        for series in series_values:
            for source in source_values:
                piece = bench_df[bench_df[series_col] == series]
                if source is not None:
                    piece = piece[piece[subgroup_col] == source]
                    label = f"{series} | {source}"
                else:
                    label = str(series)
                piece = piece.set_index("eval_subset").reindex(subsets)
                ax.bar(x + offset * width, piece[rate_col].fillna(0), width, label=label)
                offset += 1
        ax.set_xticks(x + width * max(0, offset - 1) / 2)
        ax.set_xticklabels(subsets, rotation=45, ha="right")
        ax.set_ylim(0, 1)
        ax.set_ylabel("Rate")
        ax.set_title(full_title)
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
        fig.tight_layout()
        plt.show()

## Grade Distributions

In [ ]:
include_source = analysis["scanner_source"].nunique(dropna=True) > 1

plot_grade_distributions(
    analysis,
    human_analysis,
    scanner_keys,
    human_labels_for_display,
    scanner_to_human_label=SCANNER_TO_HUMAN_LABEL,
    include_source=include_source,
)

## Violation Rates

In [ ]:
scanner_with_flags = add_violation_flags(
    analysis,
    threshold=VIOLATION_THRESHOLD,
    score_col="value_num",
    output_col="is_violation",
)
scanner_rates = summarize_violation_rates(
    scanner_with_flags,
    group_cols=["benchmark", "scanner_key", "eval_subset"] + (["scanner_source"] if include_source else []),
    flag_col="is_violation",
)

human_with_flags = add_violation_flags(
    human_analysis[human_analysis["label_name"].isin(human_labels_for_display)],
    threshold=VIOLATION_THRESHOLD,
    score_col="target_num",
    output_col="is_human_violation",
)
human_rates = summarize_violation_rates(
    human_with_flags,
    group_cols=["benchmark", "label_name", "eval_subset"],
    flag_col="is_human_violation",
)

display(scanner_rates.sort_values(["scanner_key", "eval_subset"]))
display(human_rates.sort_values(["label_name", "eval_subset"]))

# Combine scanner and human rates for unified plotting
combined_rates = pd.concat([
    scanner_rates.rename(columns={"scanner_key": "series"}),
    human_rates.rename(columns={"label_name": "series"}),
], ignore_index=True)

plot_rate_summary(
    combined_rates,
    title=f"Violation rates (threshold >= {VIOLATION_THRESHOLD})",
    series_col="series",
    subgroup_col="scanner_source" if include_source else None,
)

## Scanner Comparison And Performance Metrics

In [ ]:
scanner_comparison = summarize_scanner_comparison(
    comparison,
    scanner_keys=scanner_keys,
    violation_threshold=VIOLATION_THRESHOLD,
)
display(scanner_comparison.sort_values(["benchmark", "scanner_key", "eval_subset"]))

performance_metrics = summarize_performance_metrics(
    comparison,
    TARGET_RULES,
    scanner_keys=scanner_keys,
    violation_threshold=VIOLATION_THRESHOLD,
)
display(format_metric_percent(
    performance_metrics.sort_values(["benchmark", "scanner_key", "eval_subset"]),
    ["accuracy", "sensitivity", "specificity"],
))

for metric_name in ["accuracy", "sensitivity", "specificity"]:
    plot_rate_summary(
        performance_metrics,
        title=f"{metric_name.title()} by subset",
        series_col="scanner_key",
        rate_col=metric_name,
        subgroup_col="scanner_source" if include_source and "scanner_source" in performance_metrics.columns else None,
    )